In [70]:
conn = sq.connect("gold_analysis.db")

GDP_annual.to_sql("GDP_annual", conn, if_exists="replace", index=False)
Inflation_annual.to_sql("Inflation_annual", conn, if_exists="replace", index=False)
Gold_annual.to_sql("Gold_annual", conn, if_exists="replace", index=False)
silver_annual.to_sql("silver_annual", conn, if_exists="replace", index=False)
copper_annual.to_sql("copper_annual", conn, if_exists="replace", index=False)
macro_annual.to_sql("macro_annual", conn, if_exists="replace", index=False)

65

In [71]:
pd.read_sql("""
SELECT name
From sqlite_master
WHERE type = 'table';
""", conn)

,name
0,GDP_annual
1,Inflation_annual
2,Gold_annual
3,silver_annual
4,copper_annual
5,macro_annual


In [94]:
query_1 = """
-- Summarizes gold performance under different inflation regimes & count high and low inflation years for later analysis.
-- How: Groups the merged annual dataset by Inflation Group and calculates the number of years, average gold price, and average gold annual return for each group.
-- Purpose: This helps evaluate whether gold tends to perform differently in high-inflation versus low-inflation environments, which is central to the project’s analysis.

SELECT
    [Inflation Group] AS inflation_group,
    [GDP Growth Group] AS gdp_group,
    COUNT(*) AS year,
    ROUND(AVG([Gold Price]), 2) AS avg_gold_price,
    ROUND(AVG("gold_change_pct"), 2) AS avg_gold_change_pct
FROM macro_annual
GROUP BY [Inflation Group], [GDP Growth Group]
ORDER BY inflation_group;
"""

gold_inflation_gdp = pd.read_sql(query_1, conn)
gold_inflation_gdp

,inflation_group,gdp_group,year,avg_gold_price,avg_gold_change_pct
0,High Inflation,High GDP Growth,20,316.51,11.62
1,High Inflation,Low GDP Growth,13,718.98,14.46
2,Low Inflation,High GDP Growth,12,321.66,0.19
3,Low Inflation,Low GDP Growth,20,916.24,6.94


In [62]:
query_2 = """
-- Identifies the years with the strongest gold price increases.
-- How: Uses a window function (RANK() OVER) to rank years by gold_change_pct from highest to lowest.
-- Purpose: This helps reveal the macroeconomic conditions associated with the 10 strongest gold growth years, allowing comparison with inflation and GDP trends.

SELECT
    year,
    gold_change_pct,
    [Inflation Rate],
    gdp_growth_pct,
    RANK() OVER (ORDER BY gold_change_pct DESC) AS gold_growth_rank
FROM macro_annual
WHERE gold_change_pct IS NOT NULL
ORDER BY gold_growth_rank, year
LIMIT 10;
"""

gold_increase_rank = pd.read_sql(query_2, conn)
gold_increase_rank

,year,gold_change_pct,Inflation Rate,gdp_growth_pct,gold_growth_rank
0,1980,89.348859,13.549202,-0.256776,1
1,1973,69.136725,6.177760,5.645680,2
2,1979,63.100506,11.254471,3.165988,3
3,1974,61.753304,11.054805,-0.540550,4
4,1972,44.255492,3.272278,5.255502,5
5,2006,36.206209,3.225944,2.784540,6
6,1978,32.122077,7.630964,5.535206,7
7,2011,27.398605,3.156842,1.564407,8
8,2020,26.992511,1.233584,-2.081376,9
9,2010,25.248585,1.640043,2.695193,10


In [63]:
query_3 = """
-- Following query_2, examines the macroeconomic environment during the strongest gold growth years.
-- How: Uses a window function to rank gold returns and then retrieves their inflation group, inflation rate, and GDP growth conditions.
-- Purpose: This helps determine whether strong gold performance tends to occur under specific macroeconomic regimes, such as high inflation or weak economic growth.

SELECT
    year,
    [Inflation Group] AS inflation_group,
    [Inflation Rate] AS inflation_rate,
    [GDP Growth Group] AS gdp_growth_group,
    gdp_growth_pct,
    gold_change_pct,
    RANK() OVER (ORDER BY gold_change_pct DESC) AS gold_rank
FROM macro_annual
ORDER BY gold_rank, year
LIMIT 10;
"""

strongest_gold_inflation_gdp = pd.read_sql(query_3, conn)
strongest_gold_inflation_gdp

,year,inflation_group,inflation_rate,gdp_growth_group,gdp_growth_pct,gold_change_pct,gold_rank
0,1980,High Inflation,13.549202,Low GDP Growth,-0.256776,89.348859,1
1,1973,High Inflation,6.177760,High GDP Growth,5.645680,69.136725,2
2,1979,High Inflation,11.254471,High GDP Growth,3.165988,63.100506,3
3,1974,High Inflation,11.054805,Low GDP Growth,-0.540550,61.753304,4
4,1972,High Inflation,3.272278,High GDP Growth,5.255502,44.255492,5
5,2006,High Inflation,3.225944,Low GDP Growth,2.784540,36.206209,6
6,1978,High Inflation,7.630964,High GDP Growth,5.535206,32.122077,7
7,2011,High Inflation,3.156842,Low GDP Growth,1.564407,27.398605,8
8,2020,Low Inflation,1.233584,Low GDP Growth,-2.081376,26.992511,9
9,2010,Low Inflation,1.640043,Low GDP Growth,2.695193,25.248585,10


In [72]:
query_4 = """
-- Identifies high-inflation years in which gold outperformed its overall average annual return (15/33), combined with query_1.
-- How: Joins the gold, inflation, and merged macroeconomic tables by year, then filters for High Inflation years where gold_change_pct is greater than the overall average gold return from a subquery.
-- Purpose: This helps test whether high inflation is associated with unusually strong gold performance, which is central to evaluating gold as an inflation hedge.

SELECT
    g.year,
    m.[Inflation Group] AS inflation_group,
    i.[Inflation Rate] AS inflation_rate,
    g.[Gold Price] AS gold_price,
    m.gold_change_pct,
    m.gold_inflation_gap
FROM Gold_annual g
JOIN Inflation_annual i ON g.year = i.year
JOIN macro_annual m ON g.year = m.year
WHERE m.[Inflation Group] = 'High Inflation'
    AND m.gold_change_pct > (
        SELECT AVG(gold_change_pct) FROM macro_annual
        )
ORDER BY i.[Inflation Rate] DESC, m.gold_change_pct DESC;
"""

gold_outperform_high_inflation = pd.read_sql(query_4, conn)
gold_outperform_high_inflation

,year,inflation_group,inflation_rate,gold_price,gold_change_pct,gold_inflation_gap
0,1980,High Inflation,13.549202,607.762500,89.348859,75.799657
1,1979,High Inflation,11.254471,320.975000,63.100506,51.846035
2,1974,High Inflation,11.054805,161.679167,61.753304,50.698499
3,1978,High Inflation,7.630964,196.795833,32.122077,24.491113
4,1977,High Inflation,6.501684,148.950000,20.323124,13.821440
5,1973,High Inflation,6.177760,99.954167,69.136725,62.958965
6,1971,High Inflation,4.292767,40.966667,13.841095,9.548328
7,1968,High Inflation,4.271796,40.202500,13.855848,9.584052
8,2008,High Inflation,3.839100,880.208333,24.909828,21.070728
9,1987,High Inflation,3.664563,451.091667,22.504356,18.839793


In [73]:
query_4_2 = """
-- Identifies low-inflation years in which gold outperformed its overall average annual return (10/32), combined with query_1.
-- How: Joins the gold, inflation, and merged macroeconomic tables by year, then filters for low Inflation years where gold_change_pct is greater than the overall average gold return from a subquery.
-- Purpose: This helps test whether low inflation is associated with unusually strong gold performance, which is central to evaluating gold as an inflation hedge.

SELECT
    g.year,
    m.[Inflation Group] AS inflation_group,
    i.[Inflation Rate] AS inflation_rate,
    g.[Gold Price] AS gold_price,
    m.gold_change_pct,
    m.gold_inflation_gap
FROM Gold_annual g
JOIN Inflation_annual i ON g.year = i.year
JOIN macro_annual m ON g.year = m.year
WHERE m.[Inflation Group] = 'Low Inflation'
    AND m.gold_change_pct > (
        SELECT AVG(gold_change_pct) FROM macro_annual
        )
ORDER BY i.[Inflation Rate] DESC, m.gold_change_pct DESC;
"""

gold_outperform_low_inflation = pd.read_sql(query_4_2, conn)
gold_outperform_low_inflation

,year,inflation_group,inflation_rate,gold_price,gold_change_pct,gold_inflation_gap
0,2024,Low Inflation,2.949525,2404.577567,23.078861,20.129335
1,2007,Low Inflation,2.852672,704.675000,15.230055,12.377383
2,2004,Low Inflation,2.677237,409.808333,11.474555,8.797318
3,2003,Low Inflation,2.270095,367.625000,17.503696,15.233601
4,1986,Low Inflation,1.898048,368.225000,14.771429,12.873381
5,2019,Low Inflation,1.812210,1405.325000,11.040218,9.228008
6,2010,Low Inflation,1.640043,1230.750000,25.248585,23.608541
7,2002,Low Inflation,1.586032,312.862500,15.390644,13.804613
8,2020,Low Inflation,1.233584,1784.657500,26.992511,25.758926
9,2009,Low Inflation,-0.355546,982.645833,11.637870,11.993416


In [99]:
query_5 = """
-- Identify years when gold strongly outperformed during periods of low GDP growth.
-- How: Select years labeled 'Low GDP Growth' and keep only those where gold_change_pct exceeds the overall average gold return.
-- Purpose: This highlights unusually strong gold years in weak economic environments to test whether gold performs better when growth slows.

SELECT
    year,
    [GDP Growth Group] AS gdp_growth_group,
    gdp_growth_pct,
    gold_change_pct,
    gold_gdp_gap
FROM macro_annual
WHERE [GDP Growth Group] = 'Low GDP Growth'
    AND gold_change_pct > (
        SELECT AVG(gold_change_pct) FROM macro_annual
        )
ORDER BY gold_change_pct DESC;
"""

gold_outperform_low_gdp = pd.read_sql(query_5, conn)
gold_outperform_low_gdp

,year,gdp_growth_group,gdp_growth_pct,gold_change_pct,gold_gdp_gap
0,1980,Low GDP Growth,-0.256776,89.348859,89.605635
1,1974,Low GDP Growth,-0.540550,61.753304,62.293854
2,2006,Low GDP Growth,2.784540,36.206209,33.421669
3,2011,Low GDP Growth,1.564407,27.398605,25.834198
4,2020,Low GDP Growth,-2.081376,26.992511,29.073887
5,2010,Low GDP Growth,2.695193,25.248585,22.553392
6,2008,Low GDP Growth,0.113587,24.909828,24.796241
7,2024,Low GDP Growth,2.793187,23.078861,20.285673
8,2003,Low GDP Growth,2.795606,17.503696,14.708090
9,2002,Low GDP Growth,1.700447,15.390644,13.690197


In [100]:
query_5_2 = """
-- Identify years when gold still outperformed during periods of strong GDP growth.
-- How: Select years labeled 'High GDP Growth' and filter to those where gold_change_pct exceeds the overall average gold return.
-- Purpose: This helps compare whether strong gold years occur more often in weak economies or also during strong economic growth.

SELECT
    year,
    [GDP Growth Group] AS gdp_growth_group,
    gdp_growth_pct,
    gold_change_pct,
    gold_gdp_gap
FROM macro_annual
WHERE [GDP Growth Group] = 'High GDP Growth'
    AND gold_change_pct > (
        SELECT AVG(gold_change_pct) FROM macro_annual
        )
ORDER BY gold_change_pct DESC;
"""

gold_outperform_high_gdp = pd.read_sql(query_5_2, conn)
gold_outperform_high_gdp

,year,gdp_growth_group,gdp_growth_pct,gold_change_pct,gold_gdp_gap
0,1973,High GDP Growth,5.645680,69.136725,63.491045
1,1979,High GDP Growth,3.165988,63.100506,59.934518
2,1972,High GDP Growth,5.255502,44.255492,38.999990
3,1978,High GDP Growth,5.535206,32.122077,26.586870
4,1987,High GDP Growth,3.454630,22.504356,19.049727
5,1977,High GDP Growth,4.624187,20.323124,15.698937
6,1986,High GDP Growth,3.462655,14.771429,11.308774
7,1968,High GDP Growth,4.914509,13.855848,8.941339
8,1971,High GDP Growth,3.292722,13.841095,10.548372
9,1983,High GDP Growth,4.583791,11.508082,6.924291


In [74]:
query_6 = """
-- Compares gold prices with silver and copper prices over time.
-- How: Joins the annual gold, silver, and copper datasets by year and calculates the gold-to-silver and gold-to-copper price ratios.
-- Purpose: These ratios help evaluate whether gold behaves more similarly to another precious metal (silver) or differently from an industrial metal (copper).

SELECT
    g.year,
    g.[Gold Price],
    s.[Silver Price],
    c.[Copper Price],
    ROUND(g.[Gold Price] / NULLIF(s.[Silver Price], 0), 2) AS gold_silver_ratio,
    ROUND(g.[Gold Price] / NULLIF(c.[Copper Price], 0), 2) AS gold_copper_ratio
FROM Gold_annual g
    JOIN silver_annual s ON g.year = s.year
    JOIN copper_annual c ON g.year = c.year
ORDER BY g.year;
"""

gold_metal_ratio = pd.read_sql(query_6, conn)
gold_metal_ratio

,year,Gold Price,Silver Price,Copper Price,gold_silver_ratio,gold_copper_ratio
0,1959,35.310000,0.920000,0.310739,38.38,113.63
1,1960,35.310000,0.920000,0.298338,38.38,118.36
2,1961,35.310000,0.930000,0.298112,37.97,118.45
3,1962,35.310000,1.090000,0.292455,32.39,120.74
4,1963,35.310000,1.292250,0.297616,27.32,118.64
...,...,...,...,...,...,...
63,2022,1798.956575,21.671767,3.984608,83.01,451.48
64,2023,1953.688517,23.579675,3.873046,82.85,504.43
65,2024,2404.577567,28.128033,4.215200,85.49,570.45
66,2025,3472.540975,41.501500,4.841048,83.67,717.31


In [75]:
query_7 = """
-- Identifies years when gold outperformed both silver and copper.
-- How: Filters the macro dataset to keep only years where gold_change_pct is greater than both silver_change_pct and copper_change_pct.
-- Purpose: This helps determine when gold was the strongest-performing metal compared with other major commodities.

SELECT
    year,
    gold_change_pct,
    silver_change_pct,
    copper_change_pct
FROM macro_annual
WHERE gold_change_pct IS NOT NULL
  AND silver_change_pct IS NOT NULL
  AND copper_change_pct IS NOT NULL
  AND gold_change_pct > silver_change_pct
  AND gold_change_pct > copper_change_pct
ORDER BY gold_change_pct DESC;
"""
gold_raise_with_other_metals = pd.read_sql(query_7, conn)
gold_raise_with_other_metals

,year,gold_change_pct,silver_change_pct,copper_change_pct
0,1980,89.348859,55.537069,9.813233
1,1973,69.136725,52.626411,16.099356
2,1972,44.255492,11.800220,-1.553276
3,1978,32.122077,18.328551,-0.623320
4,2008,24.909828,10.829479,-3.034601
5,2024,23.078861,19.289317,8.834249
6,1977,20.323124,7.004414,-4.693951
7,2003,17.503696,7.455664,13.207819
8,2002,15.390644,4.977204,-1.149443
9,2007,15.230055,14.182100,4.184972


In [77]:
query_8 = """
-- Calculates the volatility (standard deviation) of annual returns for gold, silver, and copper.
-- How: Uses the variance formula based on averages of squared returns and mean returns, then applies a square root to obtain standard deviation.
-- Purpose: Volatility measures price stability, so comparing metals helps evaluate whether gold behaves as a relatively stable investment asset.

SELECT
    ROUND(
        SQRT(AVG(gold_change_pct * gold_change_pct) - AVG(gold_change_pct) * AVG(gold_change_pct)), 2
        ) AS gold_volatility,
    ROUND(
        SQRT(AVG(silver_change_pct * silver_change_pct) - AVG(silver_change_pct) * AVG(silver_change_pct)), 2
        ) AS silver_volatility,
    ROUND(
        SQRT(AVG(copper_change_pct * copper_change_pct) - AVG(copper_change_pct) * AVG(copper_change_pct)), 2
        ) AS copper_volatility
FROM macro_annual
WHERE gold_change_pct IS NOT NULL
    AND silver_change_pct IS NOT NULL
    AND copper_change_pct IS NOT NULL;
"""

gold_metal_volatility = pd.read_sql(query_8, conn)
gold_metal_volatility

,gold_volatility,silver_volatility,copper_volatility
0,21.46,28.55,21.67


In [79]:
query_9 = """
-- Compares the average annual returns of gold, silver, and copper across different macroeconomic regimes.
-- How: Groups the dataset by inflation regime and GDP growth regime, then calculates the average return of each metal within each group.
-- Purpose: This helps evaluate whether gold behaves differently from other metals under varying economic conditions.

SELECT
    [Inflation Group] as inflation_group,
    [GDP Growth Group] as gdp_growth_group,
    ROUND(AVG(gold_change_pct),2) AS avg_gold_change_pct,
    ROUND(AVG(silver_change_pct),2) AS avg_silver_change_pct,
    ROUND(AVG(copper_change_pct),2) AS avg_copper_change_pct
FROM macro_annual
WHERE gold_change_pct IS NOT NULL
    AND silver_change_pct IS NOT NULL
    AND copper_change_pct IS NOT NULL
GROUP BY [Inflation Group], [GDP Growth Group]
ORDER BY inflation_group, gdp_growth_group;
"""
metal_price = pd.read_sql(query_9, conn)
metal_price

,inflation_group,gdp_growth_group,avg_gold_change_pct,avg_silver_change_pct,avg_copper_change_pct
0,High Inflation,High GDP Growth,11.62,11.75,13.22
1,High Inflation,Low GDP Growth,14.46,13.20,4.94
2,Low Inflation,High GDP Growth,0.19,5.97,7.07
3,Low Inflation,Low GDP Growth,6.94,4.26,-0.79


In [80]:
query_10 = """
-- Compares each year’s gold price with its recent 3-year average.
-- How: Uses a self-join on the Gold_annual table to match each year with itself and the prior two years, then calculates the average price over that 3-year window.
-- Purpose: This helps identify whether gold prices are unusually high or low relative to their recent trend, which supports the analysis of longer-run movement in gold prices.

SELECT
    a.year,
    ROUND(a.[Gold Price], 2) AS current_gold_price,
    ROUND(AVG(b.[Gold Price]), 2) AS gold_3yr_avg,
    ROUND(a.[Gold Price] - AVG(b.[Gold Price]), 2) AS gold_price_diff
FROM Gold_annual a
JOIN Gold_annual b
    ON b.year BETWEEN a.year - 2 AND a.year
GROUP BY a.year, a.[Gold Price]
ORDER BY a.year DESC;
"""

gold_price_3yr_avg = pd.read_sql(query_10, conn)
gold_price_3yr_avg

,year,current_gold_price,gold_3yr_avg,gold_price_diff
0,2026,5104.77,3660.63,1444.14
1,2025,3472.54,2610.27,862.27
2,2024,2404.58,2052.41,352.17
3,2023,1953.69,1848.46,105.23
4,2022,1798.96,1792.12,6.84
...,...,...,...,...
107,1919,20.03,19.84,0.19
108,1918,19.84,19.66,0.18
109,1917,19.66,19.46,0.20
110,1916,19.47,19.36,0.11
